# 13-1절 연습 문제 풀이

이 노트북은 13-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch13/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 13장 공통 - CLIP + GPT-2 이미지 설명 생성기
# 주의: 모델과 데이터셋 내려받기가 필요하다.
try:
    from transformers import (CLIPModel, CLIPProcessor, GPT2LMHeadModel,
                              GPT2Tokenizer, Blip2Processor,
                              Blip2ForConditionalGeneration)
    from datasets import load_dataset
except ImportError:
    print('알림: pip install transformers datasets pillow 가 필요하다.')

CLIP_NAME = 'openai/clip-vit-base-patch32'
GPT2_NAME = 'gpt2'
BLIP2_NAME = 'Salesforce/blip2-opt-2.7b'

## 연습 13-1

[코드 13-1]의 단일 선형 계층 대신 다음 두 구성으로 프로젝션 계층을 다시 만들어 파라미터 수를 각각 계산해 보자.

중간 차원 1024를 두고 nn.Linear 두 개를 그대로 쌓은 구성

①의 두 nn.Linear 사이에 nn.GELU를 끼운 구성

두 프로젝션 계층 사이의 표현력에 차이가 있는지, 그리고 그 이유는 무엇인지 정리해 보자.

In [ ]:
def count(module): return sum(p.numel() for p in module.parameters())
CLIP_DIM, GPT2_DIM, MID = 512, 768, 1024

single = nn.Linear(CLIP_DIM, GPT2_DIM)
stacked = nn.Sequential(nn.Linear(CLIP_DIM, MID), nn.Linear(MID, GPT2_DIM))
with_gelu = nn.Sequential(nn.Linear(CLIP_DIM, MID), nn.GELU(), nn.Linear(MID, GPT2_DIM))

print(f'단일 선형 계층      : {count(single):,}개')
print(f'① 선형 두 개       : {count(stacked):,}개')
print(f'② 선형 + GELU + 선형: {count(with_gelu):,}개')

①과 ②의 **파라미터 수는 같다**. GELU에는 학습 파라미터가 없기 때문이다.

하지만 성능은 크게 다르다. ①처럼 선형 계층만 두 개 쌓으면 **전체가 하나의 선형 변환으로 축약**되어 단일 계층과 표현력이 같다(파라미터만 낭비). ②처럼 사이에 비선형 활성화 함수를 넣어야 비로소 더 복잡한 관계를 표현할 수 있다. 3장에서 다룬 다층 퍼셉트론의 원리와 동일하다.

## 연습 13-2

Flickr8k 데이터셋 전체(약 8천 장의 훈련 샘플)로 데이터 크기를 늘려 같은 모델을 학습해 보자. 본문 예제(200개) 대비 검증 손실과 생성 결과 품질이 어떻게 달라지는지 정리해 보자. CPU 환경에서 학습 시간이 길어질 수 있으므로 구글 코랩 등 GPU 환경 사용을 권장한다.

### 풀이

데이터를 200개에서 8,000개로 늘리면 다음이 달라진다.

- **검증 손실이 뚜렷하게 낮아진다.** 프로젝션 계층이 CLIP 임베딩 공간과 GPT-2 임베딩 공간의 대응 관계를 훨씬 안정적으로 배운다.
- **생성 캡션이 다양해진다.** 200개로 학습하면 몇 가지 문장 패턴만 반복하는데, 데이터가 늘면 이미지 내용에 따라 문장이 달라진다.
- **과적합이 줄어** 훈련 손실과 검증 손실의 격차가 좁아진다.

> **CPU 환경 주의**: 8,000장에 대해 CLIP 인코딩을 매 에포크 수행하면 매우 오래 걸린다. CLIP은 고정되어 있으므로 **이미지 특징을 한 번만 계산해 저장해 두고 재사용**하면 학습이 훨씬 빨라진다.

## 연습 13-3

[코드 13-3]의 ImageProjection에서 num_prefix_tokens를 1, 4, 16으로 각각 설정해 모델을 학습한 후, 생성 캡션의 품질을 비교해 보자. 프리픽스 토큰 수가 많아질수록 어떤 변화가 생기는지 정리해 보자.

In [ ]:
class ImageProjection(nn.Module):
    def __init__(self, clip_dim=512, gpt2_dim=768, num_prefix_tokens=4):
        super().__init__()
        self.num_prefix_tokens, self.gpt2_dim = num_prefix_tokens, gpt2_dim
        self.projection = nn.Sequential(
            nn.Linear(clip_dim, gpt2_dim), nn.LayerNorm(gpt2_dim), nn.GELU(),
            nn.Linear(gpt2_dim, gpt2_dim * num_prefix_tokens))
    def forward(self, f):
        return self.projection(f).view(-1, self.num_prefix_tokens, self.gpt2_dim)

for n_prefix in (1, 4, 16):
    m = ImageProjection(num_prefix_tokens=n_prefix)
    out = m(torch.randn(2, 512))
    print(f'프리픽스 {n_prefix:2d}개: 출력 {tuple(out.shape)}, '
          f'파라미터 {sum(p.numel() for p in m.parameters()):,}개')

프리픽스 토큰이 많아지면 이미지 정보를 담을 **자리가 늘어** 표현력이 커지지만, 파라미터도 비례해 증가한다(마지막 선형 계층이 `768 × 768 × n`).

1개면 이미지 정보를 한 벡터에 압축해야 해 정보가 부족하고, 16개면 파라미터가 많아 200개 데이터로는 과적합하기 쉽다. 데이터 규모에 맞춰 4~8개가 무난하다.

## 연습 13-4

본문 예제의 탐욕 디코딩 대신 10장에서 살펴본 빔 서치를 적용해 캡션을 생성해 보자. 빔 크기를 1(탐욕 디코딩), 3, 5로 바꿔 가며 결과를 비교해 보자.

In [ ]:
@torch.no_grad()
def generate_beam(model, projection, image_features, tokenizer,
                  beam_size=3, max_new_tokens=30):
    """프리픽스 임베딩에서 시작하는 빔 서치"""
    prefix = projection(image_features)                 # (1, P, 768)
    beams = [(prefix, [], 0.0)]                         # (임베딩, 토큰, 누적 로그확률)
    for _ in range(max_new_tokens):
        candidates = []
        for embeds, tokens, score in beams:
            if tokens and tokens[-1] == tokenizer.eos_token_id:
                candidates.append((embeds, tokens, score)); continue
            logits = model(inputs_embeds=embeds).logits[:, -1]
            logprobs = torch.log_softmax(logits, dim=-1)
            top = logprobs.topk(beam_size, dim=-1)
            for i in range(beam_size):
                tid = top.indices[0, i].item()
                emb = model.transformer.wte(torch.tensor([[tid]], device=embeds.device))
                candidates.append((torch.cat([embeds, emb], dim=1),
                                   tokens + [tid], score + top.values[0, i].item()))
        beams = sorted(candidates, key=lambda b: b[2] / max(len(b[1]), 1),
                       reverse=True)[:beam_size]
    best = max(beams, key=lambda b: b[2] / max(len(b[1]), 1))
    return tokenizer.decode(best[1], skip_special_tokens=True)

print('빔 크기 1은 탐욕 디코딩과 같다. 3~5로 늘리면 문법적으로 더 매끄러운')
print('문장이 나오지만, 무난하고 일반적인 표현으로 수렴하는 경향도 함께 커진다.')

이미지 캡셔닝에서 빔 서치는 **문법 품질을 높이지만 표현이 단조로워지는** 경향이 있다. 'a man is standing on a beach' 같은 안전한 문장이 자주 나온다.

길이 정규화(누적 로그 확률 ÷ 길이)를 빼먹으면 짧은 캡션만 선택되므로 반드시 넣어야 한다.

## 연습 13-5

[도전 문제] 본문 예제의 영어 GPT-2 대신 12장에서 사용한 한국어 LLM(예: Bllossom-3B)을 언어 모델로 사용하도록 코드를 수정해 보자. 단, 데이터셋의 이미지 설명도 한국어로 바꿔야 한다. 데이터 준비 과정에서 12장에서 소개한 Groq의 LLM API를 사용해 영어 이미지 설명을 한국어로 바꿔 사용해 보자.

### 풀이

영어 GPT-2를 한국어 LLM으로 바꾸려면 다음을 처리해야 한다.

1. **임베딩 차원 변경**: GPT-2는 768차원이지만 Bllossom-3B는 3,072차원이다. `ImageProjection`의 출력 차원을 모델의 `config.hidden_size`로 맞춘다.
2. **임베딩 조회 경로 변경**: `model.transformer.wte` 대신 `model.get_input_embeddings()`를 사용하면 모델 종류와 무관하게 동작한다.
3. **데이터 번역**: Flickr8k 캡션이 영어이므로 한국어 번역본이 필요하다. 기계 번역을 쓰거나 한국어 캡션 데이터셋(예: MS-COCO 한국어 번역본)을 사용한다.
4. **메모리**: 3B 모델은 FP16으로도 6GB 이상 필요하다. 12-3절의 4비트 양자화를 함께 적용하면 8GB VRAM에서도 가능하다.

프로젝션 계층만 학습하는 구조는 그대로 유지된다.

In [ ]:
# 모델 종류에 무관한 임베딩 조회
print('GPT-2:  model.transformer.wte(ids)')
print('일반화: model.get_input_embeddings()(ids)   <- 어떤 모델에도 동작')

## 연습 13-6

[도전 문제] 본문 예제의 구조(사전 학습된 인코더, 사전 학습된 언어 모델, 학습 가능한 작은 프로젝션 계층 조합)를 그대로 활용해, 이미지 대신 짧은 오디오 클립을 입력받아 오디오 설명 텍스트를 생성하는 모델을 만들어 보자. 오디오 인코더는 CLIP의 오디오 짝꿍 격인 CLAP 모델(laion/clap-htsat-unfused)을 사용한다. 출력이 512차원으로 CLIP과 동일해 본문 예제의 ImageProjection 코드를 거의 그대로 재사용할 수 있다. 데이터셋은 약 5천 개의 오디오와 캡션 쌍으로 구성된 Clotho를 사용한다. CPU 환경에서는 학습 시간이 길어지므로 구글 코랩 등 GPU 환경 사용을 권장한다.

In [ ]:
# CLAP(오디오) + GPT-2 구조 - CLIP을 CLAP으로 바꾸기만 하면 된다.
CLAP_NAME = 'laion/clap-htsat-unfused'
print(f'오디오 인코더: {CLAP_NAME}')
print('''
구조 변경 요약
  1) CLIPModel/CLIPProcessor -> ClapModel/ClapProcessor
  2) get_image_features(pixel_values) -> get_audio_features(input_features)
  3) 데이터셋: 이미지-캡션 쌍 -> 오디오-설명 쌍 (예: AudioCaps, Clotho)
  4) 프로젝션 계층과 GPT-2, 학습 루프는 그대로 사용
''')
print('CLAP 임베딩 차원은 512로 CLIP과 같으므로 ImageProjection을 그대로 쓸 수 있다.')

이 문제의 핵심은 **프로젝션 계층 구조가 모달에 무관하다**는 점이다. 인코더가 무엇이든 고정 크기 임베딩만 내주면, 그것을 언어 모델의 임베딩 공간으로 옮기는 방식은 동일하다.

CLAP은 CLIP과 같은 대조 학습 방식으로 오디오와 텍스트를 정렬해 두었기 때문에, 언어 모델과 잇기에 특히 유리하다.